# 03 Synthetic Indian Option Market

Generate clearly synthetic NIFTY and Bank NIFTY option chains with smile-like implied volatility, spreads, volume, and open interest.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
market = generate_synthetic_indian_market(config)
market_path = SYNTHETIC_DATA_DIR / "synthetic_nifty_banknifty_option_chain.csv"
market.to_csv(market_path, index=False)
print(f"Synthetic data written to {market_path}")
save_table(market.head(50), "03_synthetic_option_chain_sample.csv")
market.head()


In [ ]:
plt.figure()
for (symbol, maturity), group in market.groupby(["symbol", "maturity"]):
    if symbol == "NIFTY" and maturity in sorted(market[market["symbol"] == "NIFTY"]["maturity"].unique())[:3]:
        smile = group[group["option_type"] == "call"].sort_values("strike")
        plt.plot(smile["strike"] / smile["underlying_price"], smile["implied_volatility"], marker="o", label=f"{symbol} {maturity*365:.0f}d")
plt.title("Synthetic NIFTY volatility smile (not live market data)")
plt.xlabel("Strike / spot")
plt.ylabel("Implied volatility")
plt.legend()
save_current_figure("03_synthetic_nifty_volatility_smile.png")
summary = market.groupby("symbol").agg(rows=("strike", "size"), spot=("underlying_price", "first"), avg_iv=("implied_volatility", "mean"), avg_spread=("ask", lambda x: np.nan)).reset_index()
summary["avg_bid_ask_spread"] = market.groupby("symbol").apply(lambda g: (g["ask"] - g["bid"]).mean()).values
save_table(summary, "03_synthetic_market_summary.csv")
summary
